# ESC-50 Subset Sound Classification

This notebook builds an end-to-end audio classification pipeline for a subset of the ESC-50 dataset. The focus is on everyday Indonesian acoustic environments (e.g., rain, thunderstorms, animal sounds, urban sirens). We will download the data from Kaggle, prepare mel-spectrogram features, train a compact convolutional neural network (CNN), and evaluate the model with interpretability plots and sample predictions.


### Indonesian-Friendly ESC-50 Classes
We will focus on 8 classes that frequently appear in Indonesian urban/rural soundscapes:

```
['dog', 'rooster', 'rain', 'thunderstorm', 'fireworks', 'siren', 'crying_baby', 'crackling_fire']
```

Feel free to edit this list (minimum 2, maximum 8) to match the environments you want to model. The rest of the notebook automatically adapts to whichever subset you select.


### Prerequisites & Environment
1. Python 3.9+ with Jupyter (VS Code, JupyterLab, or Colab).
2. GPU is optional but speeds up training; CPU also works with longer training times.
3. Kaggle API credentials stored in `~/.kaggle/kaggle.json` for automated downloads (or manually place the ESC-50 folder under `data/`).


In [ ]:
# Install dependencies (run once per environment). Comment this cell out if everything is already installed.
%pip install -q pandas numpy matplotlib seaborn librosa soundfile scikit-learn torch torchaudio torchvision tqdm kaggle ipywidgets


In [ ]:
import os
import random
import zipfile
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

plt.style.use("seaborn-v0_8")
sns.set_theme(context="notebook")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22_050
TARGET_DURATION = 5.0  # seconds
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
MAX_SAMPLES = int(SAMPLE_RATE * TARGET_DURATION)

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0  # Windows-friendly


## Data Acquisition
The next cell downloads ESC-50 directly from Kaggle (`mmoreaux/environmental-sound-classification-50`).
- Ensure `kaggle` CLI is authenticated (`kaggle.json` in `~/.kaggle/`).
- If you already have the repository locally, place the extracted `ESC-50-master` folder inside `data/` and skip the download cell.


In [ ]:
ESC50_DATASET_SLUG = "mmoreaux/environmental-sound-classification-50"


def find_esc50_root(base_dir: Path) -> Path:
    """Return the ESC-50 root folder that contains meta/esc50.csv."""
    meta_match = list(base_dir.glob("**/meta/esc50.csv"))
    if meta_match:
        return meta_match[0].parents[1]
    candidate = base_dir / "ESC-50-master"
    if candidate.exists():
        return candidate
    raise FileNotFoundError("ESC-50 folder not found. Run download_esc50().")


def download_esc50(force: bool = False) -> Path:
    """Download ESC-50 from Kaggle using the CLI and return the extracted folder."""
    try:
        existing_root = find_esc50_root(DATA_DIR)
        if existing_root.exists() and not force:
            print(f"Found existing dataset at {existing_root}")
            return existing_root
    except FileNotFoundError:
        pass

    kaggle_dir = Path.home() / ".kaggle"
    token_path = kaggle_dir / "kaggle.json"
    if not token_path.exists():
        raise FileNotFoundError(
            "kaggle.json not found. Upload it to ~/.kaggle or manually place ESC-50 in data/."
        )

    os.chmod(token_path, 0o600)

    print("Downloading ESC-50 from Kaggle (this may take a few minutes)...")
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            ESC50_DATASET_SLUG,
            "-p",
            str(DATA_DIR),
            "-f",
            "ESC-50-master.zip",
        ],
        check=True,
    )

    downloaded = DATA_DIR / "ESC-50-master.zip"
    if not downloaded.exists():
        raise FileNotFoundError("Expected ESC-50-master.zip in data/ after download.")

    print("Extracting archive...")
    with zipfile.ZipFile(downloaded, "r") as zf:
        zf.extractall(DATA_DIR)

    print("Download + extraction complete.")
    return find_esc50_root(DATA_DIR)


# Try locating the dataset without re-downloading.
try:
    RAW_DATA_ROOT = find_esc50_root(DATA_DIR)
    print(f"ESC-50 root located at: {RAW_DATA_ROOT}")
except FileNotFoundError:
    RAW_DATA_ROOT = download_esc50()
    print(f"ESC-50 root located at: {RAW_DATA_ROOT}")


In [ ]:
META_PATH = RAW_DATA_ROOT / "meta" / "esc50.csv"
AUDIO_DIR = RAW_DATA_ROOT / "audio"

if not META_PATH.exists():
    raise FileNotFoundError(f"Metadata CSV not found at {META_PATH}")
if not AUDIO_DIR.exists():
    raise FileNotFoundError(f"Audio folder not found at {AUDIO_DIR}")

metadata = pd.read_csv(META_PATH)
print(f"Total clips in ESC-50: {len(metadata):,}")
metadata.head()


In [ ]:
SELECTED_CLASSES = [
    "dog",
    "rooster",
    "rain",
    "thunderstorm",
    "fireworks",
    "siren",
    "crying_baby",
    "crackling_fire",
]

if not (2 <= len(SELECTED_CLASSES) <= 8):
    raise ValueError("Please keep between 2 and 8 labels to satisfy the project scope.")

subset_df = metadata[metadata["category"].isin(SELECTED_CLASSES)].copy()
if subset_df.empty:
    raise ValueError("No rows match SELECTED_CLASSES. Double-check spelling against esc50.csv")

class_to_idx = {cls: idx for idx, cls in enumerate(SELECTED_CLASSES)}
idx_to_class = {v: k for k, v in class_to_idx.items()}
subset_df["target_idx"] = subset_df["category"].map(class_to_idx)

print(f"Clips kept: {len(subset_df)} across {len(SELECTED_CLASSES)} classes")
class_counts = subset_df["category"].value_counts().loc[SELECTED_CLASSES]
class_counts


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="viridis")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Clip count")
plt.title("Class distribution (ESC-50 subset)")
plt.show()


In [ ]:
def plot_waveform_and_mel(row, sample_rate: int = SAMPLE_RATE):
    file_path = AUDIO_DIR / row["filename"]
    signal, sr = librosa.load(file_path, sr=sample_rate)
    duration = len(signal) / sr

    fig, axes = plt.subplots(2, 1, figsize=(10, 6))
    times = np.linspace(0, duration, num=len(signal))
    axes[0].plot(times, signal)
    axes[0].set_title(f"Waveform: {row['category']} ({row['filename']})")
    axes[0].set_xlabel("Time [s]")
    axes[0].set_ylabel("Amplitude")

    mel = librosa.feature.melspectrogram(signal, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-Spectrogram (dB)")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()


for category in SELECTED_CLASSES[:4]:
    sample_row = subset_df[subset_df["category"] == category].sample(1, random_state=RANDOM_SEED)
    plot_waveform_and_mel(sample_row.iloc[0])


In [ ]:
def pad_or_trim(signal: np.ndarray, max_samples: int = MAX_SAMPLES) -> np.ndarray:
    if len(signal) > max_samples:
        return signal[:max_samples]
    if len(signal) < max_samples:
        pad_width = max_samples - len(signal)
        signal = np.pad(signal, (0, pad_width), mode="constant")
    return signal


def waveform_to_mel(signal: np.ndarray, sr: int = SAMPLE_RATE) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=signal,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=sr // 2,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    return mel_db


def normalize_mel(mel_db: np.ndarray) -> np.ndarray:
    mean = mel_db.mean()
    std = mel_db.std()
    return (mel_db - mean) / (std + 1e-6)


def random_time_shift(signal: np.ndarray, max_shift: int = 2_000) -> np.ndarray:
    if max_shift <= 0:
        return signal
    shift = np.random.randint(-max_shift, max_shift)
    return np.roll(signal, shift)



In [ ]:
class Esc50MelDataset(Dataset):
    def __init__(self, df: pd.DataFrame, audio_dir: Path, augment: bool = False):
        self.df = df.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        file_path = self.audio_dir / row["filename"]
        signal, _ = librosa.load(file_path, sr=SAMPLE_RATE)

        if self.augment:
            signal = random_time_shift(signal)

        signal = pad_or_trim(signal)
        mel = waveform_to_mel(signal)
        mel = normalize_mel(mel)

        mel_tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row["target_idx"], dtype=torch.long)

        return {
            "input": mel_tensor,
            "label": label,
            "category": row["category"],
            "filename": row["filename"],
        }



In [ ]:
train_df, temp_df = train_test_split(
    subset_df,
    test_size=0.3,
    stratify=subset_df["category"],
    random_state=RANDOM_SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["category"],
    random_state=RANDOM_SEED,
)

print(
    f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}"
)

train_dataset = Esc50MelDataset(train_df, AUDIO_DIR, augment=True)
val_dataset = Esc50MelDataset(val_df, AUDIO_DIR, augment=False)
test_dataset = Esc50MelDataset(test_df, AUDIO_DIR, augment=False)

common_loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": device.type == "cuda",
}

train_loader = DataLoader(train_dataset, shuffle=True, **common_loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **common_loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **common_loader_kwargs)



In [ ]:
sample_batch = next(iter(train_loader))
inputs = sample_batch["input"]
labels = sample_batch["label"]
print(f"Batch mel shape: {inputs.shape}")
print(f"Batch labels shape: {labels.shape}")

idx = 0
mel_img = inputs[idx, 0].numpy()
plt.figure(figsize=(6, 4))
librosa.display.specshow(mel_img, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel")
plt.title(f"Example mel-spectrogram → class: {idx_to_class[labels[idx].item()]}")
plt.colorbar(format="%+2.0f")
plt.show()


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, pool_size: Tuple[int, int] = (2, 2), dropout: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(pool_size),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SoundClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            ConvBlock(1, 16, dropout=0.1),
            ConvBlock(16, 32, dropout=0.15),
            ConvBlock(32, 64, dropout=0.2),
            ConvBlock(64, 128, pool_size=(2, 1), dropout=0.25),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.feature_extractor(x)
        return self.classifier(x)


model = SoundClassifier(num_classes=len(SELECTED_CLASSES)).to(device)
print(model)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f} M")


In [ ]:
from copy import deepcopy

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=3, factor=0.5, verbose=True
)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
) -> Tuple[float, float]:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in tqdm(loader, leave=False):
        inputs = batch["input"].to(device)
        labels = batch["label"].to(device)

        with torch.set_grad_enabled(is_train):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def train_model(model: nn.Module, epochs: int = EPOCHS) -> Dict[str, List[float]]:
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_state = None
    best_val_acc = 0.0

    for epoch in range(1, epochs + 1):
        print(f"Epoch {epoch}/{epochs}")
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

        scheduler.step(val_acc)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = deepcopy(model.state_dict())
            print(f"→ New best val_acc: {best_val_acc:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Loaded best model with val_acc={best_val_acc:.3f}")

    return history



In [ ]:
%%time
history = train_model(model, epochs=EPOCHS)


In [ ]:
def plot_history(history: Dict[str, List[float]]):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs_range, history["train_loss"], label="Train")
    axes[0].plot(epochs_range, history["val_loss"], label="Val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].legend()

    axes[1].plot(epochs_range, history["train_acc"], label="Train")
    axes[1].plot(epochs_range, history["val_acc"], label="Val")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_history(history)


In [ ]:
def collect_predictions(model: nn.Module, loader: DataLoader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    filenames, categories = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            inputs = batch["input"].to(device)
            labels = batch["label"].to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())
            filenames.extend(batch["filename"])
            categories.extend(batch["category"])

    return np.array(y_true), np.array(y_pred), np.array(y_prob), filenames, categories


y_true, y_pred, y_prob, test_files, test_categories = collect_predictions(model, test_loader)

test_acc = (y_true == y_pred).mean()
print(f"Test accuracy: {test_acc:.3f}")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=SELECTED_CLASSES,
        digits=4,
    )
)


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=SELECTED_CLASSES, yticklabels=SELECTED_CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix (Test Set)")
plt.show()


In [ ]:
def predict_clip(model: nn.Module, file_path: Path) -> Tuple[str, float, np.ndarray]:
    signal, _ = librosa.load(file_path, sr=SAMPLE_RATE)
    signal = pad_or_trim(signal)
    mel = waveform_to_mel(signal)
    mel = normalize_mel(mel)
    tensor = torch.tensor(mel, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = probs.argmax()
    return idx_to_class[pred_idx], probs[pred_idx], probs


def show_sample_predictions(num_samples: int = 4):
    samples = test_df.sample(num_samples, random_state=RANDOM_SEED)
    for _, row in samples.iterrows():
        file_path = AUDIO_DIR / row["filename"]
        pred_label, pred_conf, probs = predict_clip(model, file_path)
        true_label = row["category"]

        print(f"File: {row['filename']}")
        print(f"True label: {true_label}")
        print(f"Predicted: {pred_label} ({pred_conf:.2%})")

        plt.figure(figsize=(6, 3))
        mel = waveform_to_mel(pad_or_trim(librosa.load(file_path, sr=SAMPLE_RATE)[0]))
        librosa.display.specshow(mel, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel")
        plt.title(f"Mel-Spectrogram → True: {true_label} | Pred: {pred_label}")
        plt.colorbar(format="%+2.0f dB")
        plt.tight_layout()
        plt.show()

        prob_df = pd.DataFrame({"class": SELECTED_CLASSES, "probability": probs})
        prob_df = prob_df.sort_values("probability", ascending=False)
        display(prob_df)
        print("-" * 60)


show_sample_predictions(num_samples=3)


## Results & Next Steps
- Inspect the classification report/confusion matrix to find confusable pairs (e.g., `thunderstorm` vs `rain`).
- Improve performance by experimenting with: longer training, SpecAugment, pre-trained audio embeddings (e.g., PANNs, AST), or larger mel resolution.
- Deploy options: export the PyTorch model to TorchScript/ONNX, wrap inference into a FastAPI service, or integrate with a mobile edge device (Jetson, Raspberry Pi) for on-device alerting.
- Documentation: record hardware, final hyperparameters, and metrics in your final project report/paper.
